# ByteBite: Cooking-Method-Aware Nutrition Estimation from Food Images

**NSSRP Research Project - Benedictine University** | Advisor: Dr. Ghazaleh

This notebook trains and compares two models on the Google **Nutrition5k** dataset (overhead RGB images, 5 regression targets: calories, mass, fat, carbs, protein):

1. **Model 1 (control)** - a SnapNutrition-style baseline: frozen EfficientNetB3 -> GAP -> Dense(256) -> Dropout(0.3) -> Dense(5), Huber loss.
2. **Model 2 (innovation)** - the same backbone with an added auxiliary **cooking-method classification head** (labels heuristically derived from fat content). Hypothesis: learning cooking method as an auxiliary task helps the shared representation detect *hidden* oil/fat that is not directly visible in the image.

**How to run**
1. Activate the environment: `conda activate bytebite`, then launch Jupyter and open this notebook.
2. Run all cells top to bottom (Kernel -> Restart & Run All).
3. The first run downloads EfficientNetB3 ImageNet weights (~48 MB, requires internet once).
4. Everything (models, CSV results, figures, run summary) is saved to `./outputs/` next to this notebook.

**Expect:** on CPU, roughly 2-6 min/epoch per model; early stopping usually finishes each model well under the 40-epoch cap, but budget 1-4 hours total. Cell 2 prints the actual dish count found on disk (~3,260 expected).


In [1]:
# =============================================================================
# CELL 1: IMPORTS, CONFIGURATION, REPRODUCIBILITY
# =============================================================================
%matplotlib inline

import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TF C++ info spam (set BEFORE importing TF)

import sys
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")   # keep output readable; comment out while debugging

# --------------------------- Reproducibility --------------------------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)          # sets Python / NumPy / TF seeds in one call
# Optional: full op-level determinism (slower; a few ops do not support it)
# tf.config.experimental.enable_op_determinism()

# ------------------------------- Paths --------------------------------------
BASE_DIR     = Path(os.environ.get("NUTRITION5K_DIR", "nutrition5k_dataset"))
METADATA_DIR = BASE_DIR / "metadata"
IMAGERY_DIR  = BASE_DIR / "imagery" / "realsense_overhead"
CAFE1_CSV    = METADATA_DIR / "dish_metadata_cafe1.csv"
CAFE2_CSV    = METADATA_DIR / "dish_metadata_cafe2.csv"

OUTPUT_DIR = Path("outputs")               # all models / CSVs / figures are saved here
OUTPUT_DIR.mkdir(exist_ok=True)

# --------------------------- Experiment config ------------------------------
IMG_SIZE      = (192, 192)
IMG_SHAPE     = (192, 192, 3)
BATCH_SIZE    = 32
EPOCHS        = 40
PATIENCE      = 5
LEARNING_RATE = 1e-3

NUTRIENTS   = ["calories", "mass", "fat", "carb", "protein"]      # model output order
TARGET_COLS = ["total_calories", "total_mass", "total_fat", "total_carb", "total_protein"]
COOKING_CLASSES     = ["raw/other", "baked", "grilled", "fried", "sauteed"]
NUM_COOKING_CLASSES = len(COOKING_CLASSES)

# Cache decoded images in RAM (~0.4 GB as uint8 for ~3,260 images).
# Big CPU speed-up: each PNG is decoded once instead of once per epoch.
# Set to False if the machine is memory constrained.
CACHE_IN_RAM = True

# --------------------------- Environment check ------------------------------
print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
gpus = tf.config.list_physical_devices("GPU")
if len(gpus) == 0:
    print("GPU devices: 0 -> training will run on CPU (expected on this machine)")
else:
    print(f"GPU devices: {gpus}")

print("\nDataset path check:")
_missing = False
for label, p in [("cafe1 metadata", CAFE1_CSV),
                 ("cafe2 metadata", CAFE2_CSV),
                 ("imagery folder", IMAGERY_DIR)]:
    ok = p.exists()
    _missing = _missing or (not ok)
    print(f"  {label:<15}: {'OK' if ok else 'MISSING'}  {p}")
if _missing:
    raise FileNotFoundError("One or more dataset paths are missing. "
                            "Fix the paths in this cell and re-run.")
print("\nCell 1 complete: environment and paths verified.")


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# =============================================================================
# CELL 2: LOAD METADATA, FILTER ZERO-CALORIE DISHES, CROSS-REFERENCE IMAGES
# =============================================================================
# The Nutrition5k metadata CSVs are "ragged": after the first 6 dish-level
# fields, each row appends a variable number of ingredient columns, so a plain
# pd.read_csv() raises tokenizing errors. We therefore parse line-by-line and
# keep only the first 6 fields, which are always:
#   dish_id, total_calories, total_mass, total_fat, total_carb, total_protein

META_COLS = ["dish_id", "total_calories", "total_mass",
             "total_fat", "total_carb", "total_protein"]

def load_dish_metadata(csv_path):
    """Parse one ragged Nutrition5k metadata CSV into a clean 6-column DataFrame."""
    rows, skipped = [], 0
    with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.strip().split(",")
            # every valid row starts with an id like "dish_1556572657"
            if len(parts) < 6 or not parts[0].startswith("dish_"):
                skipped += 1
                continue
            try:
                rows.append([parts[0]] + [float(v) for v in parts[1:6]])
            except ValueError:                     # malformed numeric field
                skipped += 1
    df = pd.DataFrame(rows, columns=META_COLS)
    print(f"{os.path.basename(csv_path)}: parsed {len(df)} dishes "
          f"({skipped} blank/malformed lines skipped)")
    return df

meta = pd.concat([load_dish_metadata(CAFE1_CSV),
                  load_dish_metadata(CAFE2_CSV)], ignore_index=True)
print(f"\nTotal rows from both cafes            : {len(meta)}")

# Remove duplicate dish ids (keep the first occurrence)
meta = meta.drop_duplicates(subset="dish_id", keep="first").reset_index(drop=True)
print(f"After dropping duplicate dish_ids     : {len(meta)}")

# ---- Filter out zero-calorie entries (per project spec) ---------------------
meta = meta[meta["total_calories"] > 0].reset_index(drop=True)
print(f"After removing zero-calorie entries   : {len(meta)}")

# ---- Cross-reference with images actually present on disk -------------------
# Build a set of dish folders that really contain rgb.png. Set membership is
# O(1), replacing the O(n^2) nested loop in the original SnapNutrition code.
available_ids = set()
for entry in os.scandir(IMAGERY_DIR):
    if entry.is_dir() and os.path.isfile(os.path.join(entry.path, "rgb.png")):
        available_ids.add(entry.name)
print(f"Dish folders on disk with rgb.png     : {len(available_ids)}")

# Dictionary-style matching via set_index / .loc (fast, vectorised).
# sorted() makes row order deterministic (os.scandir order is arbitrary),
# which matters for a reproducible train/val/test split.
meta_indexed = meta.set_index("dish_id")
matched_ids  = sorted(set(meta_indexed.index) & available_ids)
final_df = meta_indexed.loc[matched_ids].reset_index()
final_df["image_path"] = final_df["dish_id"].map(
    lambda d: str(IMAGERY_DIR / d / "rgb.png"))

# ---- BUG FIX (SnapNutrition): define image_count BEFORE any shuffle ---------
image_count = len(final_df)
print(f"\nfinal_df: {image_count} dishes with both metadata and an rgb.png image")
print("(expected ~3,260 with the full RealSense-overhead imagery download)")

assert image_count > 0, "No dishes matched - check the dataset paths in Cell 1."
display(final_df.head())


In [ ]:
# =============================================================================
# CELL 3: EXPLORATORY DATA ANALYSIS
# =============================================================================

# ---- 3a. Summary statistics -------------------------------------------------
print("=== Summary statistics for the five regression targets ===")
display(final_df[TARGET_COLS].describe().round(2))

# ---- 3b. Distributions --------------------------------------------------------
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, col in zip(axes, TARGET_COLS):
    ax.hist(final_df[col], bins=50, color="steelblue", edgecolor="black", linewidth=0.3)
    ax.set_title(col.replace("total_", ""))
    ax.set_xlabel("kcal" if "calories" in col else "grams")
    ax.set_ylabel("count")
fig.suptitle("Distributions of the five regression targets (note the right skew)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_target_distributions.png", dpi=150)
plt.show()

# ---- 3c. Correlations -----------------------------------------------------------
corr = final_df[TARGET_COLS].corr()
short_names = [c.replace("total_", "") for c in TARGET_COLS]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(5)); ax.set_xticklabels(short_names, rotation=45)
ax.set_yticks(range(5)); ax.set_yticklabels(short_names)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax)
ax.set_title("Correlation between nutrition targets")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_target_correlations.png", dpi=150)
plt.show()

# ---- 3d. Outlier check ------------------------------------------------------------
print("=== Top-5 dishes per target (potential label outliers) ===")
for col in TARGET_COLS:
    top = final_df.nlargest(5, col)[["dish_id", col]]
    print(f"\n{col}:")
    print(top.to_string(index=False))

print("\n=== IQR outlier counts (values above Q3 + 1.5*IQR) ===")
for col in TARGET_COLS:
    q1, q3 = final_df[col].quantile([0.25, 0.75])
    hi = q3 + 1.5 * (q3 - q1)
    n_out = int((final_df[col] > hi).sum())
    print(f"{col:<16}: {n_out:>4} dishes above {hi:.1f}")
print("\nNote: outliers are KEPT (per spec, only zero-calorie dishes were removed);")
print("their impact is discussed in the limitations section of the write-up.")

# ---- 3e. Cooking-method labels (fat-based heuristic, per project spec) -------------
def infer_cooking_method(fat_grams):
    """Heuristic cooking-method label from total fat.
    3 = fried    (fat > 30 g)
    4 = sauteed  (15 g <= fat <= 30 g)
    0 = raw/other (fat < 15 g)
    Classes 1 (baked) and 2 (grilled) are reserved in the 5-way head but are
    never produced by this heuristic - see the limitations discussion."""
    if fat_grams > 30:
        return 3
    elif fat_grams >= 15:
        return 4
    else:
        return 0

final_df["cooking_method"] = final_df["total_fat"].apply(infer_cooking_method)

counts = final_df["cooking_method"].value_counts().sort_index()
print("\n=== Cooking-method label distribution ===")
for cls_idx, cnt in counts.items():
    print(f"class {cls_idx} ({COOKING_CLASSES[cls_idx]:<9}): {cnt:>5}  "
          f"({100.0 * cnt / len(final_df):.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([f"{i}\n{COOKING_CLASSES[i]}" for i in counts.index],
            counts.values, color="tan", edgecolor="black")
axes[0].set_title("Cooking-method label counts")
axes[0].set_ylabel("dishes")

palette = {0: "tab:green", 3: "tab:red", 4: "tab:orange"}
for cls in sorted(final_df["cooking_method"].unique()):
    sub = final_df[final_df["cooking_method"] == cls]
    axes[1].scatter(sub["total_fat"], sub["total_calories"], s=8, alpha=0.5,
                    color=palette.get(cls, "tab:blue"),
                    label=f"{cls}: {COOKING_CLASSES[cls]}")
axes[1].axvline(15, ls="--", c="gray"); axes[1].axvline(30, ls="--", c="gray")
axes[1].set_xlabel("total fat (g)"); axes[1].set_ylabel("total calories (kcal)")
axes[1].set_title("Fat-based heuristic boundaries (15 g / 30 g)")
axes[1].legend(markerscale=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_cooking_method_labels.png", dpi=150)
plt.show()

# ---- 3f. A few sample images --------------------------------------------------------
rng_eda = np.random.default_rng(SEED)
sample_idx = rng_eda.choice(len(final_df), size=min(8, len(final_df)), replace=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.ravel(), sample_idx):
    row = final_df.iloc[int(idx)]
    try:
        ax.imshow(plt.imread(row["image_path"]))
    except Exception as exc:
        ax.text(0.5, 0.5, f"read error:\n{exc}", ha="center", va="center", fontsize=7)
    ax.set_title(f"{row['dish_id']}\n{row['total_calories']:.0f} kcal | "
                 f"fat {row['total_fat']:.1f} g", fontsize=8)
    ax.axis("off")
fig.suptitle("Random sample of overhead RGB images")
plt.tight_layout()
plt.show()
print("Cell 3 complete: EDA finished.")


In [ ]:
# =============================================================================
# CELL 4: TRAIN / VAL / TEST SPLIT (70/15/15) + tf.data PIPELINES
# =============================================================================

# ---- 4a. Deterministic split (seed 42) --------------------------------------
# image_count was defined in Cell 2 BEFORE this shuffle (SnapNutrition bug fix).
rng = np.random.default_rng(SEED)
perm = rng.permutation(image_count)

n_train = int(0.70 * image_count)
n_val   = int(0.15 * image_count)
train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train + n_val]
test_idx  = perm[n_train + n_val:]

train_df = final_df.iloc[train_idx].reset_index(drop=True)
val_df   = final_df.iloc[val_idx].reset_index(drop=True)
test_df  = final_df.iloc[test_idx].reset_index(drop=True)
print(f"Split sizes -> train: {len(train_df)}   val: {len(val_df)}   "
      f"test: {len(test_df)}   (total {image_count})")

# Persist the exact split so every later analysis is auditable / reproducible
split_arr = np.array(["train"] * image_count, dtype=object)
split_arr[val_idx]  = "val"
split_arr[test_idx] = "test"
split_record = final_df[["dish_id"]].copy()
split_record["split"] = split_arr
split_record.to_csv(OUTPUT_DIR / "data_split_seed42.csv", index=False)
print("Saved: outputs/data_split_seed42.csv")

print("\nCooking-method label counts per split (0=raw/other, 3=fried, 4=sauteed):")
for name, df_ in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = df_["cooking_method"].value_counts().sort_index()
    print(f"  {name:<5}: " + ",  ".join(f"class {k}: {v}" for k, v in vc.items()))

# ---- 4b. Arrays for the pipelines ---------------------------------------------
def extract_arrays(df):
    paths  = df["image_path"].tolist()
    y_nut  = df[TARGET_COLS].to_numpy(dtype=np.float32)   # [cal, mass, fat, carb, protein]
    y_cook = keras.utils.to_categorical(
        df["cooking_method"].to_numpy(),
        num_classes=NUM_COOKING_CLASSES).astype(np.float32)
    return paths, y_nut, y_cook

train_paths, y_train, c_train = extract_arrays(train_df)
val_paths,   y_val,   c_val   = extract_arrays(val_df)
test_paths,  y_test,  c_test  = extract_arrays(test_df)

# ---- 4c. tf.data pipelines --------------------------------------------------------
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path):
    """Read a PNG, resize to 192x192, KEEP the raw 0-255 pixel range.
    Keras EfficientNet has its input normalisation built into the model, so
    rescaling to [0,1] here would be a silent accuracy-killing bug."""
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(tf.round(img), tf.uint8)      # uint8 keeps the RAM cache small
    return img

def make_dataset(paths, labels, shuffle=False):
    """labels: one array (Model 1) or a dict of arrays (Model 2)."""
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, y: (decode_image(p), y), num_parallel_calls=AUTOTUNE)
    if CACHE_IN_RAM:
        ds = ds.cache()                          # decode each image only once
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.map(lambda img, y: (tf.cast(img, tf.float32), y),
                num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# Model 1 datasets: target = 5 nutrition values
train_ds_m1 = make_dataset(train_paths, y_train, shuffle=True)
val_ds_m1   = make_dataset(val_paths,   y_val)
test_ds_m1  = make_dataset(test_paths,  y_test)   # NOT shuffled: row order matches y_test

# Model 2 datasets: dict targets {nutrition, cooking_method}
train_ds_m2 = make_dataset(train_paths,
                           {"nutrition": y_train, "cooking_method": c_train},
                           shuffle=True)
val_ds_m2   = make_dataset(val_paths,
                           {"nutrition": y_val, "cooking_method": c_val})
test_ds_m2  = make_dataset(test_paths,
                           {"nutrition": y_test, "cooking_method": c_test})

# ---- 4d. Sanity-check one batch ------------------------------------------------------
imgs, labs = next(iter(train_ds_m1))
pix_min, pix_max = float(tf.reduce_min(imgs)), float(tf.reduce_max(imgs))
print(f"\nBatch check -> images: {imgs.shape} {imgs.dtype}, "
      f"pixel range: {pix_min:.0f}-{pix_max:.0f}, targets: {labs.shape}")
assert pix_max > 1.5, ("Pixels look pre-normalised - EfficientNet expects raw "
                       "0-255 input (its rescaling layers are inside the model).")
print("Cell 4 complete: datasets ready.")


In [ ]:
# =============================================================================
# CELL 5: MODEL 1 - CONTROL BASELINE (SnapNutrition replica)
# =============================================================================
# EfficientNetB3 (ImageNet, frozen) -> GAP -> Dense(256, relu) -> Dropout(0.3)
# -> Dense(5, linear). Loss: Huber. Optimizer: Adam(lr=0.001).

def build_model1():
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)   # identical head initialisation across runs

    base = keras.applications.EfficientNetB3(
        include_top=False, weights="imagenet", input_shape=IMG_SHAPE)
    base.trainable = False              # frozen backbone = fixed feature extractor

    inputs = keras.Input(shape=IMG_SHAPE, name="image")
    x = base(inputs, training=False)    # training=False also freezes BatchNorm stats
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", name="shared_dense")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    outputs = layers.Dense(5, name="nutrition")(x)   # [cal, mass, fat, carb, protein]

    model = keras.Model(inputs, outputs, name="model1_control")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss=keras.losses.Huber(),
                  metrics=[keras.metrics.MeanAbsoluteError(name="mae")])
    return model

print("Building Model 1 (first run downloads EfficientNetB3 weights, ~48 MB)...")
model1 = build_model1()
try:
    model1.summary(show_trainable=True)
except TypeError:
    model1.summary()

n_trainable = int(np.sum([int(np.prod(w.shape)) for w in model1.trainable_weights]))
print(f"\nTrainable parameters (head only): {n_trainable:,}")
print(f"Steps per epoch: {int(np.ceil(len(train_df) / BATCH_SIZE))}")
print("Note: on CPU expect very roughly 2-6 minutes per epoch. Early stopping")
print("usually ends training well before the 40-epoch cap.\n")

early_stop_1 = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1)

t0 = time.time()
history1 = model1.fit(train_ds_m1,
                      validation_data=val_ds_m1,
                      epochs=EPOCHS,
                      callbacks=[early_stop_1],
                      verbose=1)
print(f"\nModel 1 training took {(time.time() - t0) / 60:.1f} min "
      f"({len(history1.history['loss'])} epochs ran)")

pd.DataFrame(history1.history).to_csv(OUTPUT_DIR / "model1_training_history.csv",
                                      index=False)
print("Saved: outputs/model1_training_history.csv")

best_epoch_1 = int(np.argmin(history1.history["val_loss"])) + 1
print(f"Best epoch by val_loss: {best_epoch_1} (best weights restored)")

h = history1.history
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="val")
axes[0].set_title("Model 1: Huber loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(h["mae"], label="train"); axes[1].plot(h["val_mae"], label="val")
axes[1].set_title("Model 1: MAE (mean over 5 targets)")
axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_model1_training.png", dpi=150)
plt.show()


In [ ]:
# =============================================================================
# CELL 6: EVALUATE MODEL 1 ON THE FULL TEST SET
# =============================================================================
# BUG FIX vs SnapNutrition "Cell 27": that notebook accidentally computed its
# results on a single 4-image batch. Here we predict on the ENTIRE unshuffled
# test dataset and assert that the prediction count equals len(test_df).

def evaluate_nutrition(y_true, y_pred, model_name):
    """Per-nutrient MAE, RMSE and MAE as a % of the nutrient mean."""
    err  = y_pred - y_true
    mae  = np.mean(np.abs(err), axis=0)
    rmse = np.sqrt(np.mean(err ** 2, axis=0))
    mean_true = y_true.mean(axis=0)
    res = pd.DataFrame({
        "nutrient": NUTRIENTS,
        "MAE": mae,
        "RMSE": rmse,
        "mean_true": mean_true,
        "MAE_pct_of_mean": 100.0 * mae / mean_true,
    })
    res.insert(0, "model", model_name)
    return res

def plot_pred_scatter(y_true, y_pred, suptitle):
    fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
    for i, (ax, name) in enumerate(zip(axes, NUTRIENTS)):
        t, p = y_true[:, i], y_pred[:, i]
        lo = min(0.0, float(p.min()))
        hi = float(max(t.max(), p.max())) * 1.05
        ax.scatter(t, p, s=6, alpha=0.35, color="steelblue")
        ax.plot([lo, hi], [lo, hi], "r--", linewidth=1)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_title(name); ax.set_xlabel("true"); ax.set_ylabel("predicted")
    fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

pred1 = np.asarray(model1.predict(test_ds_m1, verbose=1))

assert len(pred1) == len(test_df) == len(y_test), (
    f"Evaluation bug! predictions={len(pred1)} but test set={len(test_df)}")
print(f"\n[OK] Predictions computed for the FULL test set: {len(pred1)} samples "
      "(not 4 - SnapNutrition bug fixed)")

results_m1 = evaluate_nutrition(y_test, pred1, "Model 1 (control)")
display(results_m1.round(2))
print(f"Overall mean MAE across 5 nutrients: {results_m1['MAE'].mean():.2f}")

plot_pred_scatter(y_test, pred1, "Model 1: predicted vs true (full test set)")

pred_df1 = test_df[["dish_id"]].copy()
for i, n in enumerate(NUTRIENTS):
    pred_df1[f"true_{n}"] = y_test[:, i]
    pred_df1[f"pred_{n}"] = pred1[:, i]
pred_df1.to_csv(OUTPUT_DIR / "model1_test_predictions.csv", index=False)
results_m1.to_csv(OUTPUT_DIR / "model1_test_metrics.csv", index=False)
print("Saved: outputs/model1_test_predictions.csv, outputs/model1_test_metrics.csv")


In [ ]:
# =============================================================================
# CELL 7: MODEL 2 - COOKING-METHOD-AWARE MULTI-TASK MODEL
# =============================================================================
# Design note: the backbone is FROZEN, so the only trainable parameters the two
# tasks can share are in the head. Both heads therefore branch from the same
# Dense(256) representation. That shared layer is the mechanism that lets the
# auxiliary cooking-method gradients influence nutrition prediction; if the
# heads shared only the frozen backbone, the auxiliary task could not affect
# the nutrition output at all.
#
# Per the project spec the nutrition head uses MSE here, while the control uses
# Huber. NOTE: this means Model 2 differs from Model 1 in TWO ways (loss AND
# auxiliary task), which confounds attribution - see the limitations section.
# Set NUTRITION_LOSS_M2 = "huber" to run the clean single-variable ablation.
NUTRITION_LOSS_M2 = "mse"

def build_model2():
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)    # same head init as Model 1 for fairness

    base = keras.applications.EfficientNetB3(
        include_top=False, weights="imagenet", input_shape=IMG_SHAPE)
    base.trainable = False

    inputs = keras.Input(shape=IMG_SHAPE, name="image")
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    shared = layers.Dense(256, activation="relu", name="shared_dense")(x)
    shared = layers.Dropout(0.3, name="dropout")(shared)

    nutrition_out = layers.Dense(5, name="nutrition")(shared)
    cooking_out   = layers.Dense(NUM_COOKING_CLASSES, activation="softmax",
                                 name="cooking_method")(shared)

    model = keras.Model(
        inputs,
        {"nutrition": nutrition_out, "cooking_method": cooking_out},
        name="model2_cooking_aware")

    nut_loss = (keras.losses.MeanSquaredError() if NUTRITION_LOSS_M2 == "mse"
                else keras.losses.Huber())
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={"nutrition": nut_loss,
              "cooking_method": keras.losses.CategoricalCrossentropy()},
        loss_weights={"nutrition": 1.0, "cooking_method": 0.3},
        metrics={"nutrition": [keras.metrics.MeanAbsoluteError(name="mae")],
                 "cooking_method": [keras.metrics.CategoricalAccuracy(name="acc")]},
    )
    return model

model2 = build_model2()
try:
    model2.summary(show_trainable=True)
except TypeError:
    model2.summary()

n_trainable2 = int(np.sum([int(np.prod(w.shape)) for w in model2.trainable_weights]))
print(f"\nTrainable parameters (shared dense + two heads): {n_trainable2:,}\n")

early_stop_2 = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1)

t0 = time.time()
history2 = model2.fit(train_ds_m2,
                      validation_data=val_ds_m2,
                      epochs=EPOCHS,
                      callbacks=[early_stop_2],
                      verbose=1)
print(f"\nModel 2 training took {(time.time() - t0) / 60:.1f} min "
      f"({len(history2.history['loss'])} epochs ran)")

pd.DataFrame(history2.history).to_csv(OUTPUT_DIR / "model2_training_history.csv",
                                      index=False)
print("Saved: outputs/model2_training_history.csv")

# ---- training curves (metric key names vary slightly across Keras versions) ----
def find_key(hist, suffix):
    for k in hist:
        if k.endswith(suffix) and not k.startswith("val_"):
            return k
    return None

h = history2.history
key_mae, key_acc = find_key(h, "mae"), find_key(h, "acc")
n_panels = 1 + int(key_mae is not None) + int(key_acc is not None)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 3.5))
axes = np.atleast_1d(axes)
axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="val")
axes[0].set_title("Model 2: total weighted loss")
panel = 1
if key_mae is not None:
    axes[panel].plot(h[key_mae], label="train")
    axes[panel].plot(h["val_" + key_mae], label="val")
    axes[panel].set_title("Model 2: nutrition MAE")
    panel += 1
if key_acc is not None:
    axes[panel].plot(h[key_acc], label="train")
    axes[panel].plot(h["val_" + key_acc], label="val")
    axes[panel].set_title("Model 2: cooking-method accuracy")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_model2_training.png", dpi=150)
plt.show()


In [ ]:
# =============================================================================
# CELL 8: EVALUATE MODEL 2 ON THE SAME FULL TEST SET
# =============================================================================
pred2_raw = model2.predict(test_ds_m2, verbose=1)

# Keras can return a dict (named outputs) or a list depending on the version.
if isinstance(pred2_raw, dict):
    pred2_nut  = np.asarray(pred2_raw["nutrition"])
    pred2_cook = np.asarray(pred2_raw["cooking_method"])
else:
    pred2_nut, pred2_cook = np.asarray(pred2_raw[0]), np.asarray(pred2_raw[1])

assert len(pred2_nut) == len(test_df) == len(y_test), (
    f"Evaluation bug! predictions={len(pred2_nut)} but test set={len(test_df)}")
print(f"\n[OK] Model 2 evaluated on the FULL test set: {len(pred2_nut)} samples")

results_m2 = evaluate_nutrition(y_test, pred2_nut, "Model 2 (cooking-aware)")
display(results_m2.round(2))
print(f"Overall mean MAE across 5 nutrients: {results_m2['MAE'].mean():.2f}")

plot_pred_scatter(y_test, pred2_nut, "Model 2: predicted vs true (full test set)")

# ---- auxiliary cooking-method head quality ----------------------------------
cook_true = test_df["cooking_method"].to_numpy()
cook_pred = pred2_cook.argmax(axis=1)
cook_acc  = float((cook_pred == cook_true).mean())
print(f"Cooking-method head accuracy (test): {cook_acc * 100:.1f}%")

cm = pd.crosstab(pd.Series(cook_true, name="true"),
                 pd.Series(cook_pred, name="pred"))
cm.index   = [f"{i} {COOKING_CLASSES[i]}" for i in cm.index]
cm.columns = [f"{i} {COOKING_CLASSES[i]}" for i in cm.columns]
print("\nConfusion matrix (rows = true, cols = predicted):")
display(cm)

# ---- save per-sample predictions ---------------------------------------------
pred_df2 = test_df[["dish_id"]].copy()
for i, n in enumerate(NUTRIENTS):
    pred_df2[f"true_{n}"] = y_test[:, i]
    pred_df2[f"pred_{n}"] = pred2_nut[:, i]
pred_df2["true_cooking_class"] = cook_true
pred_df2["pred_cooking_class"] = cook_pred
pred_df2.to_csv(OUTPUT_DIR / "model2_test_predictions.csv", index=False)
results_m2.to_csv(OUTPUT_DIR / "model2_test_metrics.csv", index=False)
print("Saved: outputs/model2_test_predictions.csv, outputs/model2_test_metrics.csv")


In [ ]:
# =============================================================================
# CELL 9: SIDE-BY-SIDE COMPARISON - MODEL 1 vs MODEL 2 (MAE PER NUTRIENT)
# =============================================================================
comparison = pd.DataFrame({
    "nutrient":   NUTRIENTS,
    "Model1_MAE": results_m1["MAE"].to_numpy(),
    "Model2_MAE": results_m2["MAE"].to_numpy(),
})
overall = pd.DataFrame([{
    "nutrient":   "OVERALL (mean of 5)",
    "Model1_MAE": comparison["Model1_MAE"].mean(),
    "Model2_MAE": comparison["Model2_MAE"].mean(),
}])
comparison = pd.concat([comparison, overall], ignore_index=True)
comparison["improvement"]     = comparison["Model1_MAE"] - comparison["Model2_MAE"]
comparison["improvement_pct"] = (100.0 * comparison["improvement"]
                                 / comparison["Model1_MAE"])

print(f"MAE comparison on the full test set ({len(test_df)} dishes).")
print("Positive improvement = Model 2 (cooking-aware) is better.\n")
display(comparison.round(2))

comparison.to_csv(OUTPUT_DIR / "comparison_results.csv", index=False)
print("Saved: outputs/comparison_results.csv")

# grouped bar chart (per-nutrient rows only; the OVERALL row is excluded)
plot_df = comparison.iloc[:5]
x = np.arange(len(plot_df))
w = 0.38
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w / 2, plot_df["Model1_MAE"], width=w,
       label="Model 1 (control)", color="steelblue")
ax.bar(x + w / 2, plot_df["Model2_MAE"], width=w,
       label="Model 2 (cooking-aware)", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["nutrient"])
ax.set_ylabel("MAE (kcal for calories, grams otherwise)")
ax.set_title("Test-set MAE per nutrient (calories dominates the scale)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_comparison_mae.png", dpi=150)
plt.show()


In [ ]:
# =============================================================================
# CELL 10: FAT-PREDICTION ERROR - HIGH-FAT vs LOW-FAT DISHES (HYPOTHESIS TEST)
# =============================================================================
# If cooking-method awareness helps the model detect hidden oil, the fat-MAE
# improvement should be concentrated in the medium/high-fat groups, not spread
# uniformly across all dishes.

FAT_IDX = NUTRIENTS.index("fat")
CAL_IDX = NUTRIENTS.index("calories")

fat_true = y_test[:, FAT_IDX]
fat_err1 = np.abs(pred1[:, FAT_IDX]     - fat_true)
fat_err2 = np.abs(pred2_nut[:, FAT_IDX] - fat_true)
cal_err1 = np.abs(pred1[:, CAL_IDX]     - y_test[:, CAL_IDX])
cal_err2 = np.abs(pred2_nut[:, CAL_IDX] - y_test[:, CAL_IDX])

groups = [
    ("low fat (<15 g)",      fat_true < 15),
    ("medium fat (15-30 g)", (fat_true >= 15) & (fat_true <= 30)),
    ("high fat (>30 g)",     fat_true > 30),
]

rows = []
for name, mask in groups:
    n = int(mask.sum())
    if n == 0:
        print(f"WARNING: no test dishes in group: {name}")
        continue
    rows.append({
        "group": name,
        "n_dishes": n,
        "Model1_fat_MAE": fat_err1[mask].mean(),
        "Model2_fat_MAE": fat_err2[mask].mean(),
        "Model1_cal_MAE": cal_err1[mask].mean(),
        "Model2_cal_MAE": cal_err2[mask].mean(),
    })
fat_analysis = pd.DataFrame(rows)
fat_analysis["fat_MAE_improvement"] = (fat_analysis["Model1_fat_MAE"]
                                       - fat_analysis["Model2_fat_MAE"])
fat_analysis["fat_MAE_improvement_pct"] = (100.0 * fat_analysis["fat_MAE_improvement"]
                                           / fat_analysis["Model1_fat_MAE"])
display(fat_analysis.round(2))
fat_analysis.to_csv(OUTPUT_DIR / "fat_group_analysis.csv", index=False)
print("Saved: outputs/fat_group_analysis.csv\n")

for _, r in fat_analysis.iterrows():
    verdict = "IMPROVED" if r["fat_MAE_improvement"] > 0 else "did NOT improve"
    print(f"{r['group']:<22}: Model 2 {verdict} fat MAE by "
          f"{abs(r['fat_MAE_improvement']):.2f} g "
          f"({r['fat_MAE_improvement_pct']:+.1f}%)")

# ---- Bootstrap 95% CI for the fat-MAE difference on HIGH-fat dishes ---------
def bootstrap_mae_diff_ci(err_a, err_b, n_boot=2000, seed=SEED):
    """95% CI for mean(err_a) - mean(err_b). Positive => second model better."""
    rng_b = np.random.default_rng(seed)
    n = len(err_a)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng_b.integers(0, n, n)
        diffs[b] = err_a[idx].mean() - err_b[idx].mean()
    return np.percentile(diffs, [2.5, 97.5])

hi_mask = fat_true > 30
if int(hi_mask.sum()) >= 10:
    lo_ci, hi_ci = bootstrap_mae_diff_ci(fat_err1[hi_mask], fat_err2[hi_mask])
    print(f"\nHigh-fat dishes (n={int(hi_mask.sum())}): 95% bootstrap CI for the "
          f"fat-MAE improvement (Model1 - Model2) = [{lo_ci:.2f}, {hi_ci:.2f}] g")
    if lo_ci > 0:
        print("=> CI entirely above 0: the improvement on high-fat dishes is likely real.")
    elif hi_ci < 0:
        print("=> CI entirely below 0: Model 2 is likely WORSE on high-fat dishes.")
    else:
        print("=> CI includes 0: no statistically clear difference on high-fat dishes.")
else:
    print("\nToo few high-fat test dishes for a bootstrap CI.")

# ---- chart --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(fat_analysis))
w = 0.38
ax.bar(x - w / 2, fat_analysis["Model1_fat_MAE"], width=w,
       label="Model 1 (control)", color="steelblue")
ax.bar(x + w / 2, fat_analysis["Model2_fat_MAE"], width=w,
       label="Model 2 (cooking-aware)", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(fat_analysis["group"])
ax.set_ylabel("fat MAE (g)")
ax.set_title("Fat prediction error by true-fat group")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_fat_group_mae.png", dpi=150)
plt.show()

print("\nCopy the tables from Cells 9 and 10 into the research write-up below.")


In [ ]:
# =============================================================================
# CELL 11: SAVE BOTH MODELS + RUN SUMMARY
# =============================================================================
m1_path = OUTPUT_DIR / "model1_control.keras"
m2_path = OUTPUT_DIR / "model2_cooking_aware.keras"
model1.save(m1_path)
model2.save(m2_path)
print(f"Saved Model 1 -> {m1_path}  ({m1_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved Model 2 -> {m2_path}  ({m2_path.stat().st_size / 1e6:.1f} MB)")

# quick reload smoke test (proves the files are loadable later)
_reloaded = keras.models.load_model(m1_path)
print(f"Reload check: Model 1 loads OK ({_reloaded.count_params():,} parameters)")
del _reloaded

run_summary = {
    "seed": SEED,
    "n_total_dishes": int(image_count),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "img_size": list(IMG_SIZE),
    "batch_size": BATCH_SIZE,
    "model1": {
        "loss": "huber",
        "epochs_ran": len(history1.history["loss"]),
        "test_MAE": {n: float(v) for n, v in zip(NUTRIENTS, results_m1["MAE"])},
        "overall_test_MAE": float(results_m1["MAE"].mean()),
    },
    "model2": {
        "nutrition_loss": NUTRITION_LOSS_M2,
        "cooking_loss_weight": 0.3,
        "epochs_ran": len(history2.history["loss"]),
        "test_MAE": {n: float(v) for n, v in zip(NUTRIENTS, results_m2["MAE"])},
        "overall_test_MAE": float(results_m2["MAE"].mean()),
        "cooking_head_test_accuracy": float(cook_acc),
    },
}
with open(OUTPUT_DIR / "run_summary.json", "w") as f:
    json.dump(run_summary, f, indent=2)
print("Saved: outputs/run_summary.json")

print("\nAll deliverables complete. Files in ./outputs/:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("  -", p.name)


---

# Research Write-up

*(Fill in the blank fields after running the notebook end to end. Every number referenced here is printed by Cells 6, 8, 9 and 10 and saved under `outputs/`.)*

## 1. Summary of the two approaches

**Model 1 (control)** replicates the SnapNutrition architecture: a frozen, ImageNet-pretrained EfficientNetB3 backbone followed by global average pooling, a 256-unit ReLU layer with 30% dropout, and a 5-unit linear head that jointly regresses calories, mass, fat, carbohydrates and protein from a single 192x192 overhead RGB image. It trains with Huber loss and Adam (lr = 0.001), batch size 32, up to 40 epochs with early stopping (patience 5), on a fixed 70/15/15 split of the valid Nutrition5k dishes (seed 42).

**Model 2 (cooking-method-aware)** keeps the identical frozen backbone and 256-unit shared representation but adds a second 5-way softmax head that predicts a heuristic cooking-method class (raw/other, baked, grilled, fried, sauteed) alongside the nutrition head. The auxiliary labels are derived from total fat: >30 g = fried, 15-30 g = sauteed, <15 g = raw/other. The training objective is nutrition MSE + 0.3 x categorical cross-entropy. Because the backbone is frozen, the shared Dense(256) layer is the only place the two tasks interact; the hypothesis is that the auxiliary gradients push this shared representation toward visual cues of cooking (oil sheen, browning, breading) that betray hidden fat which is not directly visible as food volume.

Relative to the original SnapNutrition notebook, this implementation also fixes three engineering problems: metadata-to-image matching uses O(1) set membership and `set_index`/`.loc` lookups instead of an O(n^2) nested loop; `image_count` is computed **before** any shuffling; and evaluation runs over the **full** held-out test set with an assertion guard, rather than a single 4-image batch.

## 2. Results

| Nutrient | Model 1 MAE (control) | Model 2 MAE (cooking-aware) | Improvement | % improvement |
|---|---|---|---|---|
| calories (kcal) | ___ | ___ | ___ | ___ |
| mass (g) | ___ | ___ | ___ | ___ |
| fat (g) | ___ | ___ | ___ | ___ |
| carb (g) | ___ | ___ | ___ | ___ |
| protein (g) | ___ | ___ | ___ | ___ |
| **overall (mean)** | ___ | ___ | ___ | ___ |

Fill from `outputs/comparison_results.csv` (Cell 9). Also report MAE as a percentage of each nutrient mean (printed in Cells 6 and 8) so the numbers are comparable to the relative errors in the Nutrition5k paper, and note the cooking-method head test accuracy (Cell 8: ___%) as evidence the auxiliary task was actually learned rather than ignored.

## 3. Did cooking-method awareness improve fat prediction specifically?

Cell 10 splits the test dishes into low- (<15 g), medium- (15-30 g) and high-fat (>30 g) groups and compares each model's fat MAE (and, secondarily, calorie MAE) within each group, plus a bootstrap 95% confidence interval for the improvement on the high-fat group. The hypothesis predicts an improvement **concentrated in the medium/high-fat groups**; a uniform change across all groups (or none) would instead suggest a generic regularisation effect or no effect. Record here: high-fat fat-MAE improvement = ___ g (___%), bootstrap 95% CI = [___, ___]. If the CI excludes zero and the low-fat group shows little change, the result is consistent with the hidden-fat hypothesis; note, however, the interpretation caveat in the next section before making a causal claim about "cooking method" understanding.

## 4. Limitations (honest assessment)

The most important limitation is that the cooking-method labels are **circular**: they are derived deterministically from total fat, which is itself one of the five regression targets. The auxiliary task therefore injects no information that is not already in the labels; it re-presents fat as a coarse 3-way classification. Any observed gain supports the claim "adding a discretised-fat auxiliary objective helps the shared representation" (a legitimate finding), but not the stronger claim that the model recognises frying or sauteing as a visual technique. Relatedly, the class set is partly aspirational: baked (1) and grilled (2) are never produced by the heuristic, so two of the five softmax units are dead weight, and the three live classes are imbalanced toward raw/other.

Second, the comparison is **confounded by design**: per the project specification, Model 2's nutrition head trains with MSE while the control trains with Huber, so any difference reflects two simultaneous changes (loss function and auxiliary task). The `NUTRITION_LOSS_M2 = "huber"` switch in Cell 7 runs the clean single-variable ablation and should be reported alongside the main result. A further scale issue compounds this: all targets are regressed on raw scales, so squared calorie errors (order 10^3-10^4) numerically dwarf the cross-entropy term (order 1, weighted 0.3). The auxiliary gradient on the shared layer is consequently weak, and MSE training also tends to yield worse MAE than Huber on right-skewed targets, which may disadvantage Model 2 for reasons unrelated to the hypothesis.

Third, **capacity and data**: the frozen backbone means only the small head (~400k parameters) trains, capping both models equally and leaving the Dense(256) layer as the entire channel of task interaction. Known Nutrition5k label noise and extreme outliers were kept (only zero-calorie dishes were removed, per spec); a single overhead RGB view cannot resolve density, layering or occluded ingredients; and all data comes from two cafeterias, so generalisation to home-cooked or restaurant food is untested.

Fourth, **statistics**: this is one seed and one split. The bootstrap CI quantifies sampling uncertainty within this test set only; run-to-run training variance is unmeasured, so MAE differences of a few percent should not be over-interpreted without multi-seed replication.

## 5. Next steps

The highest-value next step is to break the label circularity: the ragged ingredient columns that Cell 2 discards contain ingredient names ("olive oil", "butter", "canola oil", "fried chicken"), from which an added-oil or cooking-method label can be mined **independently of the fat target**, directly testing the hidden-fat hypothesis. After that: run the Huber ablation of Model 2; standardise the five targets so each contributes comparably to the loss and the auxiliary weight is meaningful; unfreeze the top EfficientNet blocks and fine-tune at a low learning rate; incorporate the RealSense depth channel for portion volume; replicate over multiple seeds and report mean +/- std with the bootstrap comparison; and benchmark against the published Nutrition5k baselines. If gains persist, a small human-annotated cooking-method subset would make the auxiliary labels trustworthy.

## 6. Reproducibility

All randomness is seeded (42). The exact split is saved to `outputs/data_split_seed42.csv`; per-sample predictions, per-nutrient metrics, training histories, all figures, `run_summary.json`, and both trained models (`.keras` format, reload-tested) are written to `outputs/`.
